In [ ]:
# ── CV Scoring (HYBRID): field-check deterministik + semantik Chroma ──
# Jalankan SETELAH cv_embedding_v1 (koleksi Chroma "cv_chunks" sudah terisi).
import json, chromadb, ollama
from pathlib import Path
from datetime import date

EMBED_MODEL   = "qwen3-embedding:0.6b"
PROCESSED_DIR = Path("../data/processed")
CHROMA_DIR    = Path("../data/chroma")
COLLECTION    = "cv_chunks"
QUERY_TASK    = "Given a job requirement, retrieve the candidate experience, skills, or education that best matches it"

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
col = client.get_collection(COLLECTION)

# muat semua CV JSON (buat cek field deterministik)
CVS = {jf.stem: json.load(open(jf, encoding="utf-8")) for jf in sorted(PROCESSED_DIR.glob("*.json"))}
print(f"{len(CVS)} CV dimuat | {col.count()} vektor di Chroma")

def embed_query(text):
    return ollama.embed(model=EMBED_MODEL, input=f"Instruct: {QUERY_TASK}\nQuery:{text}")["embeddings"][0]

In [2]:
# ── 1) Helper deterministik: skills, lama pengalaman, pendidikan ──────
TODAY_YM = date.today().year * 12 + date.today().month

def candidate_skills(cv):
    s = set()
    for k in cv.get("skills", {}).get("hard_skills", []) + cv.get("skills", {}).get("soft_skills", []):
        s.add(k.lower())
    for e in cv.get("experience", []):
        for k in e.get("key_skills", []):
            s.add(k.lower())
    return s

def skill_match(required, cand):
    matched, missing = [], []
    for req in required:
        r = req.lower()
        (matched if any(r in c or c in r for c in cand) else missing).append(req)
    score = len(matched) / len(required) if required else 1.0
    return score, matched, missing

def parse_ym(s):
    if not s: return None
    p = s.split("-")
    try:
        return int(p[0]) * 12 + (int(p[1]) if len(p) > 1 else 1)
    except Exception:
        return None

def total_years(cv):
    starts, ends = [], []
    for e in cv.get("experience", []):
        st = parse_ym(e.get("start_date", ""))
        en = parse_ym(e.get("end_date", ""))
        if en is None and e.get("is_current"):
            en = TODAY_YM
        if st is not None: starts.append(st)
        if en is not None: ends.append(en)
    if not starts or not ends: return None
    return round((max(ends) - min(starts)) / 12.0, 1)

def edu_ok(required_level, cv):
    rl = required_level.lower()
    for ed in cv.get("education", []):
        if rl in (ed.get("degree", "") + " " + ed.get("field_of_study", "")).lower():
            return True
    return False

def semantic_score(job_text, cv_id, topk=3):
    res = col.query(query_embeddings=[embed_query(job_text)], n_results=topk, where={"cv_id": cv_id})
    sims = [1 - d for d in res["distances"][0]]
    return sum(sims) / len(sims) if sims else 0.0

In [3]:
# ── 2) Skor hybrid + ranking (bobot bisa diatur) ─────────────────────
WEIGHTS = {"skills": 0.40, "semantic": 0.35, "years": 0.15, "education": 0.10}

def score_cv(job, cv_id, cv):
    cand = candidate_skills(cv)
    sk, matched, missing = skill_match(job.get("required_skills", []), cand)
    sem = semantic_score(job.get("description", ""), cv_id)
    yrs = total_years(cv)

    comps = {"skills": sk, "semantic": sem}
    if job.get("min_years_experience"):
        comps["years"] = None if yrs is None else min(yrs / job["min_years_experience"], 1.0)
    if job.get("required_education"):
        comps["education"] = 1.0 if edu_ok(job["required_education"], cv) else 0.0

    tw = sum(WEIGHTS[k] for k in comps if comps[k] is not None)
    final = sum(WEIGHTS[k] * comps[k] for k in comps if comps[k] is not None) / tw
    return {"name": cv.get("personal_info", {}).get("name", cv_id), "final": final,
            "skills": sk, "semantic": sem, "years": yrs, "matched": matched, "missing": missing}

def rank_candidates(job):
    rows = sorted((score_cv(job, c, cv) for c, cv in CVS.items()), key=lambda r: r["final"], reverse=True)
    print("=" * 72)
    print(f"JOB: {job['title']}  |  wajib: {', '.join(job.get('required_skills', []))}")
    if job.get("min_years_experience"): print(f"     min pengalaman: {job['min_years_experience']} th")
    if job.get("required_education"):   print(f"     pendidikan    : {job['required_education']}")
    print("=" * 72)
    for i, r in enumerate(rows, 1):
        yrs = f"{r['years']} th" if r["years"] is not None else "? (tgl kosong)"
        print(f"#{i}  {r['name']}  ——  SKOR {r['final'] * 100:.0f}/100")
        print(f"    skills {r['skills'] * 100:3.0f}%  |  semantic {r['semantic'] * 100:3.0f}%  |  pengalaman {yrs}")
        print(f"    cocok : {', '.join(r['matched']) or '-'}")
        print(f"    kurang: {', '.join(r['missing']) or '-'}")
    return rows

In [4]:
# ── 3) Definisikan JOB DESC + jalankan ranking ───────────────────────
JOB = {
    "title": "Senior Backend Engineer",
    "required_skills": ["Java", "Spring Boot", "Microservices", "PostgreSQL", "REST API", "Docker"],
    "min_years_experience": 5,
    "required_education": "",          # kosong = tidak diwajibkan
    "required_certifications": [],
    "description": ("Merancang dan membangun microservices yang scalable, REST API berperforma tinggi, "
                    "optimasi database PostgreSQL, CI/CD, dan arsitektur backend untuk sistem enterprise."),
}
rank_candidates(JOB)

JOB: Senior Backend Engineer  |  wajib: Java, Spring Boot, Microservices, PostgreSQL, REST API, Docker
     min pengalaman: 5 th
#1  RIZA ALAMSYA  ——  SKOR 86/100
    skills 100%  |  semantic  70%  |  pengalaman ? (tgl kosong)
    cocok : Java, Spring Boot, Microservices, PostgreSQL, REST API, Docker
    kurang: -
#2  Khor Kai Xian  ——  SKOR 63/100
    skills 100%  |  semantic  45%  |  pengalaman 0.4 th
    cocok : Java, Spring Boot, Microservices, PostgreSQL, REST API, Docker
    kurang: -
#3  Wan Muhammad Alif Firdaus  ——  SKOR 53/100
    skills  50%  |  semantic  60%  |  pengalaman 2.4 th
    cocok : Java, PostgreSQL, Docker
    kurang: Spring Boot, Microservices, REST API
#4  MOHAMAD HADRI ZAMIR BIN HAIRI  ——  SKOR 43/100
    skills  33%  |  semantic  50%  |  pengalaman 2.6 th
    cocok : Java, REST API
    kurang: Spring Boot, Microservices, PostgreSQL, Docker


[{'name': 'RIZA ALAMSYA',
  'final': 0.8618355578846403,
  'skills': 1.0,
  'semantic': 0.7039333383242289,
  'years': None,
  'matched': ['Java',
   'Spring Boot',
   'Microservices',
   'PostgreSQL',
   'REST API',
   'Docker'],
  'missing': []},
 {'name': 'Khor Kai Xian',
  'final': 0.6317925403736256,
  'skills': 1.0,
  'semantic': 0.44746653238932294,
  'years': 0.4,
  'matched': ['Java',
   'Spring Boot',
   'Microservices',
   'PostgreSQL',
   'REST API',
   'Docker'],
  'missing': []},
 {'name': 'Wan Muhammad Alif Firdaus',
  'final': 0.5339687368604872,
  'skills': 0.5,
  'semantic': 0.5959196090698242,
  'years': 2.4,
  'matched': ['Java', 'PostgreSQL', 'Docker'],
  'missing': ['Spring Boot', 'Microservices', 'REST API']},
 {'name': 'MOHAMAD HADRI ZAMIR BIN HAIRI',
  'final': 0.4285831957834738,
  'skills': 0.3333333333333333,
  'semantic': 0.49826155106226605,
  'years': 2.6,
  'matched': ['Java', 'REST API'],
  'missing': ['Spring Boot', 'Microservices', 'PostgreSQL', 'Dock

In [ ]:
# ── 4) LLM REASONING: alasan kenapa tiap kandidat dapat skornya ──────
# LLM HANYA menarasikan fakta. Verdict pengalaman DIHITUNG di kode (bukan LLM)
# supaya tidak salah membandingkan angka. Grounded ke DATA -> minim halusinasi.
REASON_MODEL = "qwen2.5:7b-instruct"

def _exp_verdict(years, min_years):
    if not min_years:
        return "tidak ada syarat minimal"
    if years is None:
        return f"tidak diketahui (tahun tidak tercantum di CV; minimal {min_years} th)"
    if years >= min_years:
        return f"YA ({years} th >= {min_years} th)"
    return f"TIDAK ({years} th < {min_years} th)"

def explain_candidate(result, cv, job):
    riwayat = "\n".join(
        f'- {e.get("role","")} @ {e.get("company","")}: {e.get("summary","")}'
        for e in cv.get("experience", [])
    ) or "- (tidak ada)"
    yrs = f'{result["years"]} th' if result["years"] is not None else "tidak diketahui"
    verdict = _exp_verdict(result["years"], job.get("min_years_experience"))
    data = (
        f'KANDIDAT: {result["name"]}\n'
        f'SKOR: {round(result["final"]*100)}/100\n'
        f'SKILL COCOK: {", ".join(result["matched"]) or "-"}\n'
        f'SKILL KURANG (vs lowongan): {", ".join(result["missing"]) or "-"}\n'
        f'PENGALAMAN: {yrs}\n'
        f'MEMENUHI MIN PENGALAMAN: {verdict}\n'
        f'RINGKASAN CV: {cv.get("summary","")}\n'
        f'RIWAYAT KERJA:\n{riwayat}'
    )
    system = (
        "Tugasmu menjelaskan alasan skor kecocokan kandidat terhadap lowongan. "
        "WAJIB hanya memakai fakta di bagian DATA. DILARANG KERAS mengarang skill, perusahaan, "
        "proyek, angka, atau klaim yang tidak ada di DATA.\n"
        "ATURAN KETAT:\n"
        "- KEKUATAN: sebut skill dari SKILL COCOK + pengalaman nyata dari RIWAYAT.\n"
        "- KEKURANGAN: HANYA boleh menyebut item yang tertulis di SKILL KURANG, dan/atau jika "
        "MEMENUHI MIN PENGALAMAN = TIDAK. JANGAN PERNAH menyebut skill yang ada di SKILL COCOK sebagai kekurangan.\n"
        "- PENGALAMAN: pakai nilai field 'MEMENUHI MIN PENGALAMAN' APA ADANYA. JANGAN menghitung "
        "atau membandingkan angka tahun sendiri. Jika 'tidak diketahui', tulis bahwa lama pengalaman "
        "tidak tercantum di CV.\n"
        "- Jika SKILL KURANG '-' dan MEMENUHI MIN PENGALAMAN = YA: nyatakan semua syarat utama "
        "terpenuhi. JANGAN mengarang kekurangan.\n"
        "Jawab 2-3 kalimat Bahasa Indonesia, ringkas."
    )
    user = (
        f'LOWONGAN: {job["title"]} | butuh skill: {", ".join(job.get("required_skills", []))}'
        f' | min pengalaman: {job.get("min_years_experience", "-")} th\n\n'
        f'DATA:\n{data}'
    )
    resp = ollama.chat(
        model=REASON_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        options={"temperature": 0},
    )
    return resp["message"]["content"].strip()

# skor + ranking, lalu generate alasan tiap kandidat
_rows = sorted(
    ({**score_cv(JOB, cid, cv), "cv_id": cid} for cid, cv in CVS.items()),
    key=lambda r: r["final"], reverse=True,
)
print("===== ALASAN PER KANDIDAT (LLM, grounded ke fakta) =====")
for i, r in enumerate(_rows, 1):
    reason = explain_candidate(r, CVS[r["cv_id"]], JOB)
    print(f'\n#{i}  {r["name"]}  —  {round(r["final"]*100)}/100')
    print(f'    {reason}')